# Models for P2P-0.3-1 dataset

Two models:
1. Normal mode without LTN
2. Model with LTN



In [1]:
import arrow
import socket
from sqlalchemy.orm import Session
from tqdm.notebook import tqdm
import time
time.clock = time.time

import april
from april import Evaluator
from april.anomalydetection import *
from april.database import EventLog
from april.database import Model
from april.database import get_engine
from april.dataset import Dataset
from april.fs import DATE_FORMAT
from april.fs import get_event_log_files

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set
Creating Evaluation table


In [2]:
print(f"april was imported from {april.__file__}")

april was imported from d:\LTNcoder\april\__init__.py


In [3]:
import tensorflow as tf
physical_devices = tf.config.list_physical_devices('GPU')
print(physical_devices)
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print("GPU found")
    print("Memory growth set")
else:
    print("No GPU found")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set


# Section 1: Dataset Exploration

Dataset name: 'p2p-0.3-1'

To determine:

1. The dimensions of the 2 features of each event - name and user
2. How to divide the 2688 rows of a one hot encoded vector into understandable vectors
3. Encoder labels and numbering for them


In [4]:
dataset_name = 'p2p-0.3-1'
dataset = Dataset(dataset_name)



In [5]:
print(f"features.shape: {len(dataset.features)}")
print(f"flat_features.shape: {dataset.flat_features.shape}")
print(f"flat_onehot_features.shape: {dataset.flat_onehot_features.shape}")
print(f"flat_onehot_features_2d.shape: {dataset.flat_onehot_features_2d.shape}")


features.shape: 2
flat_features.shape: (5000, 16, 2)
flat_onehot_features.shape: (5000, 16, 168)
flat_onehot_features_2d.shape: (5000, 2688)


In [6]:
print(f"flat_onehot_features.shape: {dataset.flat_onehot_features.shape}")
print(dataset.flat_onehot_features[0])

flat_onehot_features.shape: (5000, 16, 168)


[[0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [7]:
print(dataset.flat_features[0])
print(type(dataset.flat_features))

[[ 27. 141.]
 [  7.  32.]
 [ 11.  51.]
 [  4.  27.]
 [  5. 121.]
 [ 24.  13.]
 [  9.  49.]
 [ 10. 123.]
 [  8.  40.]
 [ 26. 140.]
 [  0.   0.]
 [  0.   0.]
 [  0.   0.]
 [  0.   0.]
 [  0.   0.]
 [  0.   0.]]
<class 'numpy.ndarray'>


In [8]:
dataset.encoders

{'name': LabelEncoder(), 'user': LabelEncoder()}

In [9]:
print(dataset.encoders["name"].classes_)
# le_name_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
dataset_name_mapping = dict(zip(dataset.encoders["name"].classes_, dataset.encoders["name"].transform(dataset.encoders["name"].classes_)))
print(dataset_name_mapping)
dataset_user_mapping = dict(zip(dataset.encoders["user"].classes_, dataset.encoders["user"].transform(dataset.encoders["user"].classes_)))
print(dataset_user_mapping)
print(f"Number of names: {len(dataset.encoders['name'].classes_)}")
print(f"Number of users: {len(dataset.encoders['user'].classes_)}")
print(f"Number of One hot encoded features for each Case: {dataset.flat_onehot_features.shape[2]}")
print(f"Number of Flat features for each Case: {dataset.flat_features.shape}")


['Approve PO 1' 'Approve PO 2' 'Approve PO 3' 'Approve SC' 'Create PO'
 'Create PR' 'Create SC' 'Pay' 'Post GR' 'Post IR' 'Purchase SC'
 'Random activity 1' 'Random activity 10' 'Random activity 11'
 'Random activity 12' 'Random activity 2' 'Random activity 3'
 'Random activity 4' 'Random activity 5' 'Random activity 6'
 'Random activity 7' 'Random activity 8' 'Random activity 9' 'Release PO'
 'Release PR' '■' '▶']
{'Approve PO 1': 0, 'Approve PO 2': 1, 'Approve PO 3': 2, 'Approve SC': 3, 'Create PO': 4, 'Create PR': 5, 'Create SC': 6, 'Pay': 7, 'Post GR': 8, 'Post IR': 9, 'Purchase SC': 10, 'Random activity 1': 11, 'Random activity 10': 12, 'Random activity 11': 13, 'Random activity 12': 14, 'Random activity 2': 15, 'Random activity 3': 16, 'Random activity 4': 17, 'Random activity 5': 18, 'Random activity 6': 19, 'Random activity 7': 20, 'Random activity 8': 21, 'Random activity 9': 22, 'Release PO': 23, 'Release PR': 24, '■': 25, '▶': 26}
{'Alpha': 0, 'Alyce': 1, 'Amanda': 2, 'Anna'

In [10]:
# dataset.flat_features[:1000, 0, 1]
dataset.encoders["name"].inverse_transform(dataset.flat_features[0, :9, 0].astype(int) - 1)


array(['▶', 'Create SC', 'Purchase SC', 'Approve SC', 'Create PO',
       'Release PO', 'Post GR', 'Post IR', 'Pay'], dtype='<U18')

In [11]:
print(dataset.flat_features[0, :, 0])
print(dataset.flat_onehot_features[0, :, :27])

[27.  7. 11.  4.  5. 24.  9. 10.  8. 26.  0.  0.  0.  0.  0.  0.]
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.
  0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.


In [12]:
print(dataset.flat_onehot_features_2d.shape)

(5000, 2688)


In [13]:
print(dataset.attribute_dims)
print(dataset.attribute_types)
print(dataset.attribute_keys)

[ 27. 141.]
[<AttributeType.CATEGORICAL: 0>, <AttributeType.CATEGORICAL: 0>]
['name', 'user']


In [14]:
print(np.sum(dataset.attribute_dims))


168.0


# Section 2: Simple DAE model and Evalution

To do:

1. Create a simple DAE model (Human understandability of the output - similar of LTN not required)
2. Carry out evalutions
   1. MSE
   2. Softmax (How?) Needs Thresholding

## Model

In [15]:
def fit_and_save(dataset_name, ad, ad_kwargs=None, fit_kwargs=None):
    if ad_kwargs is None:
        ad_kwargs = {}
    if fit_kwargs is None:
        fit_kwargs = {}

    # Save start time
    start_time = arrow.now()

    # Dataset
    if isinstance(dataset_name, str):
        dataset = Dataset(dataset_name)
    else:
        dataset:april.Dataset = dataset_name
        dataset_name = dataset.dataset_name

    # AD
    ad = ad(**ad_kwargs)

    # Train and save
    ad.fit(dataset, **fit_kwargs)
    file_name = f'{dataset_name}_{ad.abbreviation}_{start_time.format(DATE_FORMAT)}'
    model_file = ad.save(file_name)

    # Save end time
    end_time = arrow.now()

    # Cache result
    print(model_file.str_path)
    results = Evaluator(model_file.str_path).cache_result()

    # Calculate training time in seconds
    training_time = (end_time - start_time).total_seconds()

    # Write to database
    engine = get_engine()
    session = Session(engine)

    session.add(Model(creation_date=end_time.datetime,
                      algorithm=ad.name,
                      training_duration=training_time,
                      file_name=model_file.file,
                      training_event_log_id=EventLog.get_id_by_name(dataset_name),
                      training_host=socket.gethostname(),
                      hyperparameters=str(dict(**ad_kwargs, **fit_kwargs))))
    session.commit()
    session.close()

    if isinstance(ad, NNAnomalyDetector):
        from keras.backend import clear_session
        clear_session()
    
    return results

In [16]:
# from may.models.p2pdaeltn import P2PDAE, P2PDAELTN

In [17]:
results_p2pdae = fit_and_save(dataset, P2PDAE, fit_kwargs=dict(epochs=20, batch_size=100))

Epoch 1/20
45/45 [==============================] - 1s 8ms/step - loss: 0.2080 - accuracy: 0.0147 - val_loss: 0.2029 - val_accuracy: 0.2520
Epoch 2/20
45/45 [==============================] - 0s 6ms/step - loss: 0.0427 - accuracy: 0.2287 - val_loss: 0.0550 - val_accuracy: 0.8560
Epoch 3/20
45/45 [==============================] - 0s 5ms/step - loss: 0.0089 - accuracy: 0.3664 - val_loss: 0.0247 - val_accuracy: 0.7760
Epoch 4/20
45/45 [==============================] - 0s 5ms/step - loss: 0.0065 - accuracy: 0.4516 - val_loss: 0.0167 - val_accuracy: 0.7820
Epoch 5/20
45/45 [==============================] - 0s 5ms/step - loss: 0.0059 - accuracy: 0.4753 - val_loss: 0.0129 - val_accuracy: 0.7620
Epoch 6/20
45/45 [==============================] - 0s 5ms/step - loss: 0.0057 - accuracy: 0.4787 - val_loss: 0.0106 - val_accuracy: 0.7280
Epoch 7/20
45/45 [==============================] - 0s 5ms/step - loss: 0.0055 - accuracy: 0.5029 - val_loss: 0.0091 - val_accuracy: 0.6700
Epoch 8/20
45/45 [==

In [18]:
print(results_p2pdae)

In [19]:
# ads = [
#     dict(ad=P2PDAE, fit_kwargs=dict(epochs=30, batch_size=500)),
# ]
# for ad in ads:
#     [fit_and_save(d, **ad) for d in tqdm([dataset_name], desc=ad['ad'].name, leave=True, position=1)]

## Evaluation

# Section 3: Minimal LTN Model and Evaluation

To do:

1. Create a Basic LTN Predicated Model
2. Create Basic LTN Axioms
3. Carry out Evaluations:
   1. MSE
   2. Softmax (How?) Needs thresholding

## LTN Model

In [20]:
# ltn_model = P2PDAELTN(dataset, epochs=30, batch_size=500)

## LTN Axioms

In [21]:
# Writtten in the module file

## Evaluation

In [22]:
# def fit_and_save_p2pdaeltn(dataset_name, ad, ad_kwargs=None, fit_kwargs=None):
#     if ad_kwargs is None:
#         ad_kwargs = {}
#     if fit_kwargs is None:
#         fit_kwargs = {}

#     # Save start time
#     start_time = arrow.now()

#     # Dataset
#     if isinstance(dataset_name, str):
#         dataset = Dataset(dataset_name)
#     else:
#         dataset:april.Dataset = dataset_name
#         dataset_name = dataset.dataset_name

#     # AD
#     ad = ad(**ad_kwargs)

#     # Train and save
#     ad.fit(**fit_kwargs)
#     file_name = f'{dataset_name}_{ad.abbreviation}_{start_time.format(DATE_FORMAT)}'
#     model_file = ad.save(file_name)

#     # Save end time
#     end_time = arrow.now()

#     # Cache result
#     print(model_file.str_path)
#     results =  Evaluator(model_file.str_path).cache_result()

#     # Calculate training time in seconds
#     training_time = (end_time - start_time).total_seconds()

#     # Write to database
#     engine = get_engine()
#     session = Session(engine)

#     session.add(Model(creation_date=end_time.datetime,
#                       algorithm=ad.name,
#                       training_duration=training_time,
#                       file_name=model_file.file,
#                       training_event_log_id=EventLog.get_id_by_name(dataset_name),
#                       training_host=socket.gethostname(),
#                       hyperparameters=str(dict(**ad_kwargs, **fit_kwargs))))
#     session.commit()
#     session.close()

#     if isinstance(ad, NNAnomalyDetector):
#         from keras.backend import clear_session
#         clear_session()
#     return results

In [23]:
# results = fit_and_save(dataset, P2PDAELTN)


In [24]:
# print(results)

## Evaluation

In [25]:
# Save start time
start_time = arrow.now()

# AD
my_p2pdaeltn = P2PDAELTN(dataset)

# Train and save
my_p2pdaeltn.fit()
file_name = f'{dataset_name}_{my_p2pdaeltn.abbreviation}_{start_time.format(DATE_FORMAT)}'
model_file = my_p2pdaeltn.save(file_name)

# Save end time
end_time = arrow.now()

# Cache result
print(model_file.str_path)
results =  Evaluator(model_file.str_path).cache_result()

# Calculate training time in seconds
training_time = (end_time - start_time).total_seconds()

# Write to database
engine = get_engine()
session = Session(engine)

session.add(Model(creation_date=end_time.datetime,
                    algorithm=my_p2pdaeltn.name,
                    training_duration=training_time,
                    file_name=model_file.file,
                    training_event_log_id=EventLog.get_id_by_name(dataset_name),
                    training_host=socket.gethostname(),
                    hyperparameters=""
                    ))
session.commit()
session.close()

if isinstance(my_p2pdaeltn, NNAnomalyDetector):
    from keras.backend import clear_session
    clear_session()

Epoch 1/20
45/45 [==============================] - 1s 7ms/step - loss: 0.2075 - val_loss: 0.1961
Epoch 2/20
45/45 [==============================] - 0s 7ms/step - loss: 0.0418 - val_loss: 0.0503
Epoch 3/20
45/45 [==============================] - 0s 6ms/step - loss: 0.0090 - val_loss: 0.0226
Epoch 4/20
45/45 [==============================] - 0s 6ms/step - loss: 0.0065 - val_loss: 0.0153
Epoch 5/20
45/45 [==============================] - 0s 6ms/step - loss: 0.0060 - val_loss: 0.0118
Epoch 6/20
45/45 [==============================] - 0s 6ms/step - loss: 0.0057 - val_loss: 0.0097
Epoch 7/20
45/45 [==============================] - 0s 6ms/step - loss: 0.0055 - val_loss: 0.0085
Epoch 8/20
45/45 [==============================] - 0s 5ms/step - loss: 0.0053 - val_loss: 0.0076
Epoch 9/20
45/45 [==============================] - 0s 5ms/step - loss: 0.0053 - val_loss: 0.0070
Epoch 10/20
45/45 [==============================] - 0s 5ms/step - loss: 0.0052 - val_loss: 0.0065
Epoch 11/20
45/45 [

In [26]:
row1 = dataset.flat_onehot_features_2d[0].reshape(1,-1)
print(row1.shape)
indiv_pred_row1 = my_p2pdaeltn.individual_models[0].predict(row1)
print(indiv_pred_row1)
# softmax of indiv_pred_row1
indiv_pred_row1 = tf.nn.softmax(indiv_pred_row1, axis=1)
print(indiv_pred_row1)

(1, 2688)
1/1 [==============================] - 0s 79ms/step
[[0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
  8.545824e-38 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
  0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
  0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
  0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
  0.000000e+00 1.000000e+00]]
tf.Tensor(
[[0.03482102 0.03482102 0.03482102 0.03482102 0.03482102 0.03482102
  0.03482102 0.03482102 0.03482102 0.03482102 0.03482102 0.03482102
  0.03482102 0.03482102 0.03482102 0.03482102 0.03482102 0.03482102
  0.03482102 0.03482102 0.03482102 0.03482102 0.03482102 0.03482102
  0.03482102 0.03482102 0.09465335]], shape=(1, 27), dtype=float32)


In [27]:
# my_p2pdaeltn.model.predict(row1)
row1 = ltn.Variable("row1", row1)
const_26 = ltn.Constant(26, trainable=False)
my_p2pdaeltn.individual_predicates[0]([row1, const_26])

ltn.Formula(tensor=[0.09465335], free_vars=['row1'])

In [28]:
# ads = [
#     dict(ad=DAE, fit_kwargs=dict(epochs=60, batch_size=500))
# ]
# for ad in ads:
#     [fit_and_save(d, **ad) for d in tqdm(datasets, leave=True, position=1)]

In [29]:
print("Done")

Done
